In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
B = 5
T = 8
d_k = 4
w = 4 
Q = torch.randint(0,2,(B,T,d_k),dtype = float)
K = torch.randint(0,2,(B,T,d_k),dtype = float)
V = torch.randint(0,2,(B,T,d_k),dtype = float)
mask_curr = torch.triu(torch.ones(w,w), diagonal=1).bool()
mask_prev= torch.triu(torch.ones(w,w), diagonal=1).bool()

In [ ]:
Q_chunks = [Q[:,j:j+w,:].float() for j in range(0,T,w)] #Q_chunk[0] : (B,w,d_k)
K_chunks = [K[:,j:j+w,:].float() for j in range(0,T,w)]  #K_chunks.T(-2,-1) : (B,d_k,w) 
# @ = ( B,w,d_k) @ (B,d_k,w) -> (B,w,w) @(B,w,d_k) -> (B,w,d_k)
V_chunks = [V[:,j:j+w,:].float() for j in range(0,T,w)]
chunk_curr = tuple([(F.softmax((Q_chunks[i]@K_chunks[i].transpose(-2,-1)/d_k**0.5).masked_fill(mask_curr, float('-inf')),dim=-1,dtype = torch.float)@V_chunks[i]) for i in range(len(Q_chunks))])
res = torch.cat(chunk_curr,dim=1).nan_to_num(0) 
res.shape

torch.Size([5, 8, 4])